> 在线讲义：[第 23 章 · 复杂度：为什么 attention 是 n²，长文本为什么难](https://wikieden.github.io/transformer-visual/lessons/23-complexity.html) · 本书《从零读懂 Transformer》

# 第 23 章 · 复杂度：为什么 attention 是 n²，长文本为什么难

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/23-complexity.html) · 仓库里 `transformer-deep-dive/notes/lessons/23-complexity.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| 打分的乘法次数 | $n^2 d$ | `FlopCounterMode` 数 `q @ k.T` | d = 8，n = 4、8、16：128、512、2048 |
| FLOPs | 乘一次、加一次各算 1 个 | `counter.get_total_flops()` | 256、1024、4096 = 2 × 乘法 |
| 和 RNN 比 | $n^2d$ vs $nd^2$ | 直接算 | n = 50、d = 512：1,280,000 vs 13,107,200 |
| RNN 要排队 | 一步等一步 | `nn.RNNCell` 的 for 循环 | 6 个词算 6 步 |
| 分数表的内存 | $n^2$ × 2 字节 | `numel() * element_size()` | n = 1000、1 万、10 万：2 MB、200 MB、20 GB |

只算讲义里的例子（n = 4、8、16；n = 50；n = 1000、10000、100000），练习题里的数留给你自己算。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch
from torch import nn
from torch.utils.flop_counter import FlopCounterMode

**函数 `score_flops`**：让 PyTorch 自己数 Q @ K.T 用了多少 FLOPs（一次乘 + 一次加算 2 个）。

In [ ]:
def score_flops(n: int, d: int) -> int:
    """让 PyTorch 自己数 Q @ K.T 用了多少 FLOPs（一次乘 + 一次加算 2 个）。"""
    q, k = torch.randn(n, d), torch.randn(n, d)
    with FlopCounterMode(display=False) as counter:
        q @ k.T
    return counter.get_total_flops()

**函数 `rnn_steps`**：RNN 传话：第 t 步要等第 t − 1 步的结果，只能一步一步 for 循环。返回排队了几步。

In [ ]:
def rnn_steps(n: int, d: int) -> int:
    """RNN 传话：第 t 步要等第 t − 1 步的结果，只能一步一步 for 循环。返回排队了几步。"""
    cell = nn.RNNCell(d, d)
    x, h = torch.randn(n, d), torch.zeros(d)
    steps = 0
    with torch.no_grad():
        for t in range(n):
            h = cell(x[t], h)  # 依赖上一步的 h，没法同时算
            steps += 1
    return steps

**函数 `score_table_bytes`**：一层、一个头的 n × n 分数表，按半精度每个数 2 字节。

In [ ]:
def score_table_bytes(n: int, bytes_per_number: int = 2) -> int:
    """一层、一个头的 n × n 分数表，按半精度每个数 2 字节。"""
    return n * n * bytes_per_number

**函数 `human`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def human(nbytes: int) -> str:
    for unit, size in [("GB", 1e9), ("MB", 1e6), ("KB", 1e3)]:
        if nbytes >= size:
            return f"{nbytes / size:.0f} {unit}"
    return f"{nbytes} B"

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("打分 Q @ K.T，d = 8（FlopCounterMode 数的是 FLOPs = 2 × 乘法次数）")
for n in [4, 8, 16]:
    flops = score_flops(n, 8)
    print(f"  n = {n:>2}: 分数表 {n * n:>3} 格，FLOPs {flops:>5}，乘法 {flops // 2:>5} = n² × d")
n, d = 50, 512
print(f"\nn = {n}、d = {d}：attention 打分 n²d = {n * n * d:,}，RNN 每层 nd² = {n * d * d:,}")
print(f"  比值 = n ÷ d = {n * n * d / (n * d * d):.3f}")
print(
    "\nRNN 要排队：6 个词的 for 循环算了",
    rnn_steps(6, 8),
    "步，每步都等上一步（中间传话 5 次）；attention 一次 Q @ K.T 算完整张表",
)

In [ ]:
print("\n一张分数表的内存（半精度）")
real = torch.empty(1000, 1000, dtype=torch.float16)
print(f"  n = 1,000 实际分配: {human(real.numel() * real.element_size())}")
for n in [10_000, 100_000]:
    print(f"  n = {n:,}（只算不分配）: {human(score_table_bytes(n))}")

## 3. 代码里才看得到的

1. **FLOPs**（floating point operations：浮点 / 运算 次数）：论文和工具数计算量时，乘一次、加一次各算 1 个，所以是乘法次数的 2 倍。以后看到"训练用了多少 FLOPs"就是这个单位（续篇数 FLOPs 那章会用）。
2. **RNN 只能写成 for 循环**：`h = cell(x[t], h)` 右边要用上一步的 `h`，没法让第 5 步和第 4 步同时算；attention 的 `q @ k.T` 一次交给 GPU。这就是 Table 1 里"排队步数 O(n) vs O(1)"。
3. **只算不分配**：n = 10 万时一张表 20 GB，真分配会把 Mac 内存撑爆，所以代码里只用乘法算大小。

## 4. 自己改一改

- 练习题（n = 10、20，2000，4000 → 32000）先手算，再改代码里的 n 验证。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 全班两两握手，n 乘 n 格，每格 d 次乘，n 方乘 d。
> 句子长一倍，握手变四倍。
> RNN 传话要排队，路长信号丢；attention 同时握手，一步直达。
> n 比 d 小它更省，n 比 d 大平方咬人。
> 长文本三条路：只看邻居、分块不存、换掉 softmax。